In [84]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from model import PositionalEncoding
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import math

In [85]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h, dropout=0.1):
        super().__init__()
        assert d_model % h == 0, "d_model must be divisible by num_heads"

        self.h = h
        self.d_k = d_model // h

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        n_samples, seq_len, d_model = x.size()

        # (n_samples, seq_len, d_model)
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # (n_samples, seq_len, h, d_k)
        Q = Q.view(n_samples, seq_len, self.h, self.d_k)
        K = K.view(n_samples, seq_len, self.h, self.d_k)
        V = V.view(n_samples, seq_len, self.h, self.d_k)

        # Swap h and seq_len, matmul only works on last 2 dimensions
        # (n_samples, h, seq_len, d_k)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        qkt = Q @ K.transpose(2, 3)  # (n_samples, h, seq_len, seq_len)
        scores = qkt / math.sqrt(self.d_k)

        # Fills elements where mask is True to -inf
        if mask is not None:
            # (seq_len, seq_len) -> (1, 1, seq_len, seq_len) for batch, n_samples broadcasting
            mask = mask.unsqueeze(0).unsqueeze(0).to(x.device)
            scores = scores.masked_fill(mask, float('-inf'))

        weights = self.dropout(F.softmax(scores, dim=-1))

        output = weights @ V  # (n_samples, h, seq_len, d_k)

        # (n_samples, seq_len, d_model)
        concat = output.transpose(1, 2).contiguous().view(
            n_samples, seq_len, d_model)

        return self.W_O(concat)

In [54]:
~torch.tril(torch.ones((4, 4), dtype=torch.bool))

tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

In [ ]:
def fib(n_samples, seq_len, max_value):
    dataset = []
    for _ in range(n_samples):
        a, b = torch.randint(1, 10, (2,)).tolist()
        seq = [a, b]
        for _ in range(seq_len - 2):
            next_val = seq[-1] + seq[-2]
            if next_val > max_value:
                break
            seq.append(next_val)

        pad = [0] * (seq_len - len(seq))
        dataset.append(seq + pad)

    return dataset


fib(3, 10, 100)

[[2, 3, 5, 8, 13, 21, 34, 55, 89, 0],
 [2, 3, 5, 8, 13, 21, 34, 55, 89, 0],
 [9, 8, 17, 25, 42, 67, 0, 0, 0, 0]]

In [ ]:
class FibDataset(Dataset):
    def __init__(self, n_samples, seq_len, max_value):
        self.data = fib(n_samples, seq_len, max_value)

    def __getitem__(self, index):
        seq = torch.Tensor(self.data[index]).long()

        X = seq[:-1]
        Y = seq[1:]

        return X, Y

    def __len__(self): return len(self.data)

In [ ]:
fibds = FibDataset(10, 10, 100)

fibds[6]

(tensor([ 2,  1,  3,  4,  7, 11, 18, 29, 47]),
 tensor([ 1,  3,  4,  7, 11, 18, 29, 47, 76]))

In [72]:
train_dataset = FibDataset(20000, 10, 99)
val_dataset = FibDataset(2000, 10, 99)
test_dataset = FibDataset(2000, 10, 99)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False
)

In [86]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, h, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, h, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        seq_len = x.shape[1]

        mask = ~torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool))

        mha_out = self.mha(x, mask)
        x = self.norm1(x + self.dropout(mha_out))

        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))

        # (batch, seq_len, dims)
        return x

In [87]:
class TransformerFib(nn.Module):
    def __init__(self, vocab_size, d_model, seq_len, num_heads, num_layers):
        super().__init__()

        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(seq_len, d_model)
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, num_heads * 4) for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.emb(x)
        x = self.pos_encoding(x)

        for layer in self.layers:
            x = layer(x)

        x = self.fc_out(x)
        return x

In [94]:
model = TransformerFib(
    vocab_size=100,
    d_model=128,
    seq_len=9,
    num_heads=4,
    num_layers=4
)
device = 'cpu'

In [89]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)


def train(model, dataloader):
    model.train()
    total_loss = 0

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        logits = model(x)

        loss = criterion(logits.view(-1, 100), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [90]:
def validate(model, dataloader):
    model.eval()
    total_loss = 0

    # to calculate accuracy
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            # inputs = (batch, seqlen)
            # targets = (batch, seqlen)
            inputs, targets = inputs.to(device), targets.to(device)

            # (batch_size, seq len, vocabsize)
            logits = model(inputs)

            # CrossEntropyLoss expects (batch, num_classes)
            loss = criterion(logits.view(-1, 100), targets.view(-1))

            # (batch_size, seqlen)
            predictions = torch.argmax(logits, dim=-1)
            total_loss += loss.item()
            correct += (predictions == targets).sum().item()
            total += targets.numel()

    return total_loss / len(dataloader), (correct / total) * 100

In [95]:
epochs = 5
best_val_loss = float('inf')
for epoch in range(epochs):
    train_loss = train(
        model,
        train_loader,
    )
    val_loss, val_acc = validate(
        model,
        val_loader,
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')

    print(
        f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

Epoch 01 | Train Loss: 4.7571 | Val Loss: 4.7474 | Val Acc: 1.21%
Epoch 02 | Train Loss: 4.7576 | Val Loss: 4.7474 | Val Acc: 1.21%
Epoch 03 | Train Loss: 4.7583 | Val Loss: 4.7474 | Val Acc: 1.21%


KeyboardInterrupt: 

In [92]:
def infer(model, input, seq_len):
    model.eval()
    seq = torch.Tensor(input).long()

    for _ in range(seq_len - 2):
        with torch.no_grad():
            output = model(seq)

            next_token_logits = output[0, -1, :]
            next_token = torch.argmax(next_token_logits).item()
            next_token_tensor = torch.Tensor([next_token]).long()

            seq = torch.cat([seq, next_token_tensor], dim=1)

            if next_token == 0:
                break

    return seq

In [93]:
infer(model, [1, 1], 10)

RuntimeError: The size of tensor a (2) must match the size of tensor b (9) at non-singleton dimension 1